# Backward by hand: one small network, any `w` and `A`

A two-layer network, written as a PyTorch module, with its backward pass written by hand. Every policy-gradient
method in [corrections.md](corrections.md) changes only the gradient on the logits:

```python
ds2 = w * A * (p - y_onehot) / N        # (B, S, V)
```

Everything below the logits (`dw2`, `dw1`) is the same code for every method. So `backward` takes `w` and `A` as
inputs: REINFORCE, PPO, CISPO and TIS are four different `w` tensors passed to the same function.

**Table 1. How to think about a backward pass.**

| Idea | Rule |
|---|---|
| Names | `s` = pre-activation (the weighted sum), `a` = activation (after the nonlinearity), `d` + name = its gradient. `x -> s1 -> a1 -> s2` forward, `ds2 -> dw2, da1 -> ds1 -> dw1` backward. |
| Shapes | The gradient of a tensor has that tensor's shape. If you forget a rule, match the shapes. |
| Matmul `s = a @ W` | `dW = a.T @ ds` (the layer's input times the gradient coming back); `da = ds @ W.T` (sent to the layer below). |
| Activation `a = f(s)` | `ds = da * f'(s)`, evaluated where the forward pass was. ReLU: `(s > 0)`. |
| Cache | The forward pass saves each matmul's input (`x`, `a1`) and each activation's input (`s1`). That is all backward needs. |
| `gather` vs one-hot | The loss needs one number per position, the log-prob of the written token: `gather` picks it. The gradient needs a full `(V,)` row per position, `p - y_onehot`: the one-hot builds it. |
| The update | `W -= lr * dW`. On the logits, the written token's entry is `w * A * (p[y] - 1) / N`: negative when `A > 0`, so subtracting it raises that logit. |

Sections: 1 setup, 2 the network, 3 check against autograd, 4 what `ds2` looks like, 5 passing `w` and `A`,
6 several steps on one rollout.

## 1. Setup

One prompt, `B = 4` sampled responses, `S = 3` tokens each, vocabulary `V = 5`. Only response 0 got the reward.
`float64` so the hand gradients match autograd to the last digit.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
torch.set_printoptions(precision=3, sci_mode=False)
torch.manual_seed(0)

B, S, I, H, V = 4, 3, 8, 16, 5        # responses, tokens per response, input size, hidden size, vocabulary

x = torch.randn(B, S, I)              # (B, S, I)  the input at each position
y = torch.randint(0, V, (B, S))       # (B, S)     token ids the engine wrote (the labels)
R = torch.tensor([1.0, 0.0, 0.0, 0.0])                  # (B,)  reward per response
A_grpo = (R - R.mean()) / R.std()                       # (B,)  GRPO advantage, one per response
print("y:\n", y)
print("A:", A_grpo)

y:
 tensor([[1, 0, 2],
        [4, 2, 2],
        [2, 4, 4],
        [3, 0, 1]])
A: tensor([ 1.500, -0.500, -0.500, -0.500])


## 2. The network

`w` and `A` can come as a number, one value per response `(B,)`, or one value per token `(B, S)`. `per_token`
turns any of these into `(B, S)`, so `loss` and `backward` never care which one you passed.

`loss` is the policy-gradient loss `-(w * A * log p[y]).mean()`. With `w = A = 1` it is plain cross-entropy.
`backward` computes the gradient of exactly that loss by hand. It is split in two: `logits_grad` is the one line
that depends on the method, `backward_from_logits` is the part every method shares.

In [2]:
def per_token(v, B, S):
    # A number, (B,) or (B, S) -> (B, S). One value per response is copied to every token of that response.
    v = torch.as_tensor(v, dtype=torch.get_default_dtype())
    if v.dim() == 1:
        v = v.reshape(B, 1)                                        # (B, 1)
    return v.expand(B, S)                                          # (B, S)  expand stretches size-1 dims, no copy


class NN(nn.Module):
    def __init__(self, input_dim, hidden_dim, vocab_size):
        super().__init__()
        # nn.Parameter registers a tensor as a weight, so model.parameters(), .to(device) and state_dict() see it.
        # Dividing by sqrt(fan_in) (the layer's input size) keeps each layer's outputs about size 1.
        self.w1 = nn.Parameter(torch.randn(input_dim, hidden_dim) / input_dim ** 0.5)     # (I, H)
        self.w2 = nn.Parameter(torch.randn(hidden_dim, vocab_size) / hidden_dim ** 0.5)   # (H, V)

    # ---------------------------------------------------------------- forward
    def forward(self, x):
        s1 = x @ self.w1                     # (B, S, H)  pre-activation
        a1 = s1 * (s1 > 0)                   # (B, S, H)  activation (ReLU)
        s2 = a1 @ self.w2                    # (B, S, V)  logits
        logp = torch.log_softmax(s2, -1)     # (B, S, V)  log-probabilities; never takes log of a rounded-to-0 p
        p = logp.exp()                       # (B, S, V)  probabilities, y_hat in corrections.md
        # Cache for backward: each matmul's input (x, a1), the ReLU's input (s1), and p for the loss gradient.
        self.acts = {'x': x, 's1': s1, 'a1': a1, 'p': p}
        return logp

    # ---------------------------------------------------------------- loss: gather picks one value per position
    def loss(self, logp, y, w=1.0, A=1.0):
        B, S, V = logp.shape
        w, A = per_token(w, B, S), per_token(A, B, S)                 # (B, S), (B, S)
        logp_y = logp.gather(-1, y.unsqueeze(-1)).squeeze(-1)        # (B, S)  logp_y[b, s] = logp[b, s, y[b, s]]
        return -(w.detach() * A * logp_y).mean()                      # ()      w is a frozen weight: no gradient

    # ---------------------------------------------------------------- backward, step 1: the method's line
    @torch.no_grad()
    def logits_grad(self, y, w=1.0, A=1.0):
        p = self.acts['p']                                            # (B, S, V)
        B, S, V = p.shape
        w, A = per_token(w, B, S), per_token(A, B, S)                 # (B, S), (B, S)
        y_onehot = F.one_hot(y, V)                                    # (B, S, V)  a full row per position
        # -log p[y] differentiated through softmax is (p - y_onehot). w * A scales it per position.
        # / (B * S) because the loss is a mean over B * S positions.
        return w.unsqueeze(-1) * A.unsqueeze(-1) * (p - y_onehot) / (B * S)      # (B, S, V)

    # ---------------------------------------------------------------- backward, step 2: shared by every method
    @torch.no_grad()
    def backward_from_logits(self, ds2):
        x, s1, a1 = self.acts['x'], self.acts['s1'], self.acts['a1']
        I, H = self.w1.shape
        V = self.w2.shape[1]
        # s2 = a1 @ w2: weight grad = input.T @ grad; grad for the layer below = grad @ w2.T.
        # reshape(-1, .) merges B and S, so the matmul also sums over all B * S positions.
        dw2 = a1.reshape(-1, H).T @ ds2.reshape(-1, V)                # (H, V)
        da1 = ds2 @ self.w2.T                                         # (B, S, H)
        # a1 = relu(s1): pass the gradient only where the unit was on in the forward pass.
        ds1 = da1 * (s1 > 0)                                          # (B, S, H)
        # s1 = x @ w1: same matmul rule. x is the input layer's "activation".
        dw1 = x.reshape(-1, I).T @ ds1.reshape(-1, H)                 # (I, H)
        self.grads = {'w1': dw1, 'w2': dw2}
        return self.grads

    def backward(self, y, w=1.0, A=1.0):
        return self.backward_from_logits(self.logits_grad(y, w, A))

    # ---------------------------------------------------------------- update: step against the gradient
    @torch.no_grad()
    def step(self, lr):
        self.w1 -= lr * self.grads['w1']                              # (I, H)
        self.w2 -= lr * self.grads['w2']                              # (H, V)

## 3. Check against autograd

Autograd differentiates `loss`; the hand backward never looks at it. If the two agree for cross-entropy
(`w = A = 1`) and for a random `w` and `A`, the hand backward is the gradient of that loss.

In [3]:
def check(model, x, y, w=1.0, A=1.0):
    # Hand gradients vs autograd, for the same w and A.
    model.zero_grad()
    logp = model(x)                                                   # (B, S, V)
    model.loss(logp, y, w, A).backward()                              # autograd fills model.w1.grad, model.w2.grad
    g = model.backward(y, w, A)                                       # by hand
    return {k: torch.allclose(g[k], getattr(model, k).grad) for k in g}

model = NN(I, H, V)
logp = model(x)                                                       # (B, S, V)
print("loss at init:", round(model.loss(logp, y).item(), 3), "  log(V):", round(torch.log(torch.tensor(V)).item(), 3))
print("same as F.cross_entropy:", torch.allclose(model.loss(logp, y), F.cross_entropy(logp.reshape(-1, V), y.reshape(-1))))
print("cross-entropy (w = A = 1):      ", check(model, x, y))
print("w per token, A per response:     ", check(model, x, y, w=torch.rand(B, S) * 2, A=A_grpo))
print("w per response, A per token:     ", check(model, x, y, w=torch.rand(B), A=torch.randn(B, S)))

loss at init: 2.07   log(V): 1.609
same as F.cross_entropy: True
cross-entropy (w = A = 1):       {'w1': True, 'w2': True}
w per token, A per response:      {'w1': True, 'w2': True}
w per response, A per token:      {'w1': True, 'w2': True}


The loss at init is close to `log(V) = 1.609`, the loss of a model that spreads its probability evenly over the
`V` tokens.

## 4. What `ds2` looks like

One position of response 0 (`A > 0`) and one of response 1 (`A < 0`), with `w = 1`. Each row sums to 0: the update
moves probability between tokens.

In [4]:
logp = model(x)                                                       # (B, S, V)
ds2 = model.logits_grad(y, A=A_grpo) * (B * S)                        # (B, S, V)  times B * S to show it per position
print("response 0, token 0: y =", y[0, 0].item(), " p =", model.acts['p'][0, 0].detach())
print("   ds2 =", ds2[0, 0], "  written token negative: step raises it; the rest go down")
print("response 1, token 0: y =", y[1, 0].item(), " p =", model.acts['p'][1, 0].detach())
print("   ds2 =", ds2[1, 0], "  written token positive: step lowers it; the rest go up")
print("row sums:", ds2.sum(-1)[0, 0].item(), ds2.sum(-1)[1, 0].item())

response 0, token 0: y = 1  p = tensor([0.091, 0.278, 0.362, 0.264, 0.005])
   ds2 = tensor([ 0.136, -1.084,  0.543,  0.396,  0.008])   written token negative: step raises it; the rest go down
response 1, token 0: y = 4  p = tensor([0.154, 0.094, 0.190, 0.451, 0.111])
   ds2 = tensor([-0.077, -0.047, -0.095, -0.225,  0.445])   written token positive: step lowers it; the rest go up
row sums: 1.1102230246251565e-16 5.551115123125783e-17


## 5. Passing `w` and `A`

`A` comes from the rewards (section 1, `(B,)`). `w` comes from two ratios, both per token `(B, S)`, as in
[corrections.md](corrections.md) section 1:

```python
r   = p_now / p_old        # (B, S)  staleness: trainer now vs trainer when the rollout was made
rho = p_old / p_engine     # (B, S)  mismatch: trainer vs engine on the same weights
```

Each method is a small function `(r, rho, A) -> w`. The rollout stores `logp_old` (the trainer's log-prob of each
written token) and `logp_engine` (the engine's). The engine is simulated here as the trainer plus noise.

In [5]:
def logp_written(logp, y):
    return logp.gather(-1, y.unsqueeze(-1)).squeeze(-1)               # (B, S)  log-prob of each written token

def w_reinforce(r, rho, A):
    return torch.ones_like(r)                                         # (B, S)

def w_ppo(r, rho, A):
    w = r.clone()                                                     # (B, S)
    w[(A > 0) & (r > 1.28)] = 0                                       # moved up far enough: stop
    w[(A < 0) & (r < 0.8)] = 0                                        # moved down far enough: stop
    return w

def w_cispo(r, rho, A):
    return r.clamp(0, 4)                                              # (B, S)  never 0, at most 4

def w_tis(r, rho, A):
    return rho.clamp(0, 2) * w_ppo(r, rho, A)                         # (B, S)  PPO times the capped engine weight

METHODS = {'reinforce': w_reinforce, 'ppo': w_ppo, 'cispo': w_cispo, 'tis': w_tis}

torch.manual_seed(1)
rollout_model = NN(I, H, V)
with torch.no_grad():
    logp_old = logp_written(rollout_model(x), y)                      # (B, S)  stored at rollout, never updated
logp_engine = logp_old + 0.3 * torch.randn(B, S)                      # (B, S)  simulated engine log-probs
A = per_token(A_grpo, B, S)                                           # (B, S)

At the rollout weights `r = 1`, so REINFORCE, PPO and CISPO give the same `w`; TIS differs by `rho`. PPO's real loss is
`-min(r * A, clip(r) * A)`, not the frozen-`w` loss above, yet its gradient is `w_ppo * A * (p - y_onehot) / N`.
The check below runs it one step away from the rollout weights, where `r` is no longer 1.

In [6]:
def ratios(model):
    logp_now = logp_written(model(x), y)                              # (B, S)  also caches acts for backward
    r = (logp_now - logp_old).exp()                                   # (B, S)
    rho = (logp_old - logp_engine).exp()                              # (B, S)
    return logp_now, r, rho

model = NN(I, H, V)
model.load_state_dict(rollout_model.state_dict())
_, r, rho = ratios(model)
print("at the rollout weights, r =\n", r.detach())
print("w, tis:\n", w_tis(r, rho, A).detach())

model.backward(y, w_reinforce(r, rho, A), A)                          # one step away from the rollout weights
model.step(lr=0.5)

model.zero_grad()
logp_now, r, rho = ratios(model)
ppo_loss = -torch.minimum(r * A, r.clamp(0.8, 1.28) * A).mean()      # PPO as written in the paper, autograd
ppo_loss.backward()
g = model.backward(y, w_ppo(r.detach(), rho, A), A)                  # by hand, w = w_ppo
print("after one step, r =\n", r.detach())
print("PPO paper loss vs hand backward with w = w_ppo:", {k: torch.allclose(g[k], getattr(model, k).grad) for k in g})

at the rollout weights, r =
 tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
w, tis:
 tensor([[0.849, 1.421, 1.151],
        [1.629, 1.034, 0.619],
        [1.288, 1.370, 0.755],
        [0.851, 0.631, 1.232]])
after one step, r =
 tensor([[2.852, 1.326, 1.145],
        [0.703, 0.927, 1.043],
        [0.973, 0.712, 0.581],
        [0.679, 0.830, 1.089]])
PPO paper loss vs hand backward with w = w_ppo: {'w1': True, 'w2': True}


## 6. Several steps on one rollout

Each method trains a fresh copy of the rollout model on the same rollout. Each step: forward, compute `r` and `rho`,
`w = method(r, rho, A)`, `backward(y, w, A)`, `step`. Only the `w` line differs between methods.

Columns: `p(y)` averaged over response 0 (good, should rise) and over responses 1 to 3 (bad, should fall), and the
share of tokens with `w = 0` at the last step.

In [7]:
def train(method, steps=10, lr=0.5):
    model = NN(I, H, V)
    model.load_state_dict(rollout_model.state_dict())
    for _ in range(steps):
        with torch.no_grad():
            logp_now, r, rho = ratios(model)                          # (B, S) each
        w = METHODS[method](r, rho, A)                                # (B, S)  the only method-specific line
        model.backward(y, w, A)
        model.step(lr)
    p_y = logp_written(model(x), y).exp().detach()                    # (B, S)
    return {'p(y) good': round(p_y[0].mean().item(), 3),
            'p(y) bad': round(p_y[1:].mean().item(), 3),
            'w = 0': round((w == 0).double().mean().item(), 2)}

p0 = logp_old.exp()                                                   # (B, S)
print("rollout:   ", {'p(y) good': round(p0[0].mean().item(), 3), 'p(y) bad': round(p0[1:].mean().item(), 3)})
print("\n".join(f"{m:10s} {train(m)}" for m in METHODS))

rollout:    {'p(y) good': 0.184, 'p(y) bad': 0.243}
reinforce  {'p(y) good': 0.884, 'p(y) bad': 0.224, 'w = 0': 0.0}
ppo        {'p(y) good': 0.267, 'p(y) bad': 0.198, 'w = 0': 0.92}
cispo      {'p(y) good': 0.923, 'p(y) bad': 0.125, 'w = 0': 0.0}
tis        {'p(y) good': 0.284, 'p(y) bad': 0.19, 'w = 0': 0.92}


**Reading the result.** REINFORCE and CISPO keep training every token, so the good response's `p(y)` keeps rising.
PPO zeroes a token once its `r` leaves `[0.8, 1.28]` in the direction `A` wants; after a few steps most tokens are
out of the band and training on this rollout stops (the trust region). TIS behaves like PPO here, since it is PPO's
`w` times the engine weight. The responses share the same weights and the inputs are random, so pushing one token
moves the others: read the numbers as the direction of each method, not as a benchmark.

The methods differed only in the `w` tensor passed to `backward`. To add one, write `(r, rho, A) -> w` and add it to
`METHODS`. To change the advantage, pass a different `A`: a number, `(B,)` or `(B, S)`.